# 03 · Experimentos (Colab T4)

Corre el diseño factorial completo (LoRA y DoRA) con reanudación. **Forzar GPU T4** (Entorno de ejecución → Cambiar tipo de entorno → T4) para que todas las corridas sean comparables. Cada corrida guarda su resultado apenas termina; si Colab se desconecta, vuelve a ejecutar esta celda y continúa donde quedó.

In [ ]:
# === Setup (en Colab se ejecuta solo; en local no hace nada) ===
import os, sys
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/triaje-clinico-lora"):
        !git clone https://github.com/Hammonne/triaje-clinico-lora.git /content/triaje-clinico-lora
    %cd /content/triaje-clinico-lora
    !git pull -q
    !pip install -q -r requirements-colab.txt
    !pip install -q -e . --no-deps
    # Versiones usadas (para reportar en el informe)
    !pip list 2>/dev/null | grep -iE "^(torch|transformers|peft|bitsandbytes|accelerate) "

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, las salidas van a Drive para sobrevivir desconexiones.
if EN_COLAB:
    config.drive_dir = Path("/content/drive/MyDrive/triaje")
    config.results_dir = config.drive_dir / "results"
    config.output_dir  = config.drive_dir / "adaptadores"
    config.results_dir.mkdir(parents=True, exist_ok=True)
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name, "| resultados en:", config.results_dir)

## Verificar GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No hay GPU. Activa la T4 en Colab."
print("GPU:", torch.cuda.get_device_name(0))

## (Opcional) Smoke-test en CPU antes de gastar GPU

Verifica que el pipeline corre con un modelo diminuto y pocos ejemplos.

In [ ]:
# from triaje.config import Config
# from triaje.experimentos import correr_suite
# smoke = Config().aplicar_smoke_test()
# correr_suite(smoke, rs_piloto=(), rs_factorial=(4,), ratios=(1,), semillas=(42,), incluir_piloto=False)

## Correr la suite completa (piloto + factorial, LoRA y DoRA, 2 réplicas)

In [ ]:
from triaje.experimentos import correr_suite
correr_suite(config, incluir_baseline=True)   # reanuda automáticamente